In [1]:
# 6.2 langgraph实现Multi-Agent Systems

In [8]:
import http.client
import json
import requests
import json

from langchain_core.messages import (
BaseMessage,
HumanMessage,
ToolMessage,
)
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from typing import Literal
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END,StateGraph,MessagesState
from langgraph.prebuilt import ToolNode

# 导⼊聊天提示模板和消息占位符
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 导⼊状态图相关的常量和类
from langgraph.graph import END, StateGraph, START

In [9]:
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(model_name="deepseek-chat",
                 api_key=API_KEY,base_url="https://api.deepseek.com")

# 定义⼀个函数，⽤于创建代理
def create_agent(llm, tools, system_message: str):
     """创建⼀个代理。"""
     # 创建⼀个聊天提示模板
     prompt = ChatPromptTemplate.from_messages(
         [
             (
             "system",
             "你是⼀个有帮助的AI助⼿，与其他助⼿合作。"
             " 使⽤提供的⼯具来推进问题的回答。"
             " 如果你不能完全回答，没关系，另⼀个拥有不同⼯具的助⼿"
             " 会接着你的位置继续帮助。执⾏你能做的以取得进展。"
             " 如果你或其他助⼿有最终答案或交付物，"
             " 在你的回答前加上FINAL ANSWER，以便团队知道停⽌。"
             " 你可以使⽤以下⼯具: {tool_names}。\n{system_message}",
             ),
             # 消息占位符
             MessagesPlaceholder(variable_name="messages"),
         ]
     )
     # 传递系统消息参数
     prompt = prompt.partial(system_message=system_message)
     # 传递⼯具名称参数
     prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
     # 绑定⼯具并返回提示模板
     return prompt | llm.bind_tools(tools)

In [10]:
# #定义工具函数
# @tool
# def get_search_result(question):
#     """
#     互联网搜索函数
#     :param question: 必要参数，字符串类型，用于表示在互联网上进行搜素的关键词或者搜索内容的简短描述，\
#     :return：SerpAPI API根据参数question进行互联网搜索后的结果，其中包含了全部重要的搜索结果内容。
#     """
#     from langchain_community.utilities import SerpAPIWrapper
#     serpapi_api_key = "60f286e601f44a26600e42c65e7a9b3ceb06a3f0dc8e0fe7ce56ec93d6274ccd"
#     search = SerpAPIWrapper(serpapi_api_key=serpapi_api_key)
#     result = search.run(question)
#     return result

#定义工具函数
# import http.client
# import json
@tool
def get_search_result(question):
    """
    互联网搜索函数
    :param question: 必要参数，字符串类型，用于表示在互联网上进行搜素的关键词或者搜索内容的简短描述，\
    :return：SerpAPI API根据参数question进行互联网搜索后的结果，其中包含了全部重要的搜索结果内容。
    """
    url = "https://google.serper.dev/search"
    payload = json.dumps({
      "q": question,
      "hl": "zh-cn"
    })
    headers = {
      'X-API-KEY': '2694386d405bc7b92d36d41897accf7d27c3e2c1',
      'Content-Type': 'application/json'
    }
    response = requests.request("POST", url, headers=headers, data=payload)
    print(response.text)
    return response.text

#定义工具函数，用于Agent调用外部工具
@tool
def send_email(query:str):
    """邮件发送工具，可以接受query内容，然后进行邮件发送"""
    return "邮件已成功发送。"

In [11]:
#定义工具节点
# 导⼊预构建的⼯具节点
from langgraph.prebuilt import ToolNode
# 定义⼯具列表
tools = [get_search_result, send_email]
# 创建⼯具节点
tool_node = ToolNode(tools)

#定义状态：我们⾸先定义图的状态。这只是⼀个消息列表，以及⼀个⽤于跟踪最新发送者的键
# 导⼊操作符和类型注解
import operator
from typing import Annotated, Sequence, TypedDict
# 导⼊OpenAI聊天模型
from langchain_openai import ChatOpenAI


In [12]:
# 定义⼀个对象，⽤于在图的每个节点之间传递
# 我们将为每个代理和⼯具创建不同的节点
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    sender: str

In [13]:
#定义代理节点
import functools
from langchain_core.messages import AIMessage

# ⽤于为给定的Agent创建节点
def agent_node(state, agent, name):#name:agent代理的名字
    # 调⽤代理
    result = agent.invoke(state)
    #将 result 转换为 AIMessage 类型，并进行进一步处理
    #使用模型的 model_dump 方法将 result 转换为字典格式，同时排除 "type" 和 "name" 字段。这通常用于序列化对象以便传输或存储。
    result = AIMessage(**result.model_dump(exclude={"type", "name"}), name=name)
    return {
        "messages": [result],
        # 由于我们有⼀个严格的⼯作流程，我们可以跟踪发送者，以便知道下⼀个传递给谁。
        "sender": name,
    }

In [14]:
#创建搜索Agent代理对象和节点对象
research_agent = create_agent(
    llm,
    [get_search_result],
    system_message="你应该提供准确的数据供MailOpt使⽤。",
)
# 创建Agent节点对象：使用agent和name的值填充到agent_node函数中对应的两个参数
research_node = functools.partial(agent_node, agent=research_agent, name="Researcher")


#创建发邮件Agent代理对象和节点对象
mail_agent = create_agent(
    llm,
    [send_email],
    system_message="你用于进行邮件发送业务实现",
)
# 创建Agent节点对象
mail_node = functools.partial(agent_node, agent=mail_agent, name="MailOpt")

In [15]:
#定义路由函数，决定是否继续执行
from typing import Literal
# 定义路由器函数,continue 表示代理应该继续处理消息队列中的下一条消息。
def router(state) -> Literal["call_tool", "__end__", "continue"]:
    # 这是路由器
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        # 上⼀个代理正在调⽤⼯具
        return "call_tool"
    if "FINAL ANSWER" in last_message.content:
        # 任何代理决定⼯作完成
        return "__end__"
    return "continue"

In [16]:
#图、节点和边的创建
# 创建状态图实例
workflow = StateGraph(AgentState)
# 添加搜索节点
workflow.add_node("Researcher", research_node)
# 添加邮件节点
workflow.add_node("MailOpt", mail_node)
# 添加⼯具调⽤节点
workflow.add_node("call_tool", tool_node)

# 添加条件边
workflow.add_conditional_edges(
     "Researcher",
     router,
     {"continue": "MailOpt", "call_tool": "call_tool", "__end__": END},
)
workflow.add_conditional_edges(
     "MailOpt",
     router,
     {"continue": "Researcher", "call_tool": "call_tool", "__end__": END},
)
# 添加条件边
workflow.add_conditional_edges(
     "call_tool",
     #如果 x["sender"] 的值是 "Researcher"，那么边会连接到 "Researcher" 节点。
	 #如果 x["sender"] 的值是 "MailOpt"，那么边会连接到 "MailOpt" 节点。
     lambda x: x["sender"],
     {
         "Researcher": "Researcher",
         "MailOpt": "MailOpt",
     },
)
# 添加起始边
workflow.add_edge(START, "Researcher")
# 编译⼯作流图
graph = workflow.compile()

# 将⽣成的图⽚保存到⽂件
graph_png = graph.get_graph().draw_mermaid_png()
with open("collaboration.png", "wb") as f:
    f.write(graph_png)

In [17]:
#调用
events = graph.invoke(
    {
        "messages": [
            HumanMessage(
            content="获取过去5年AI软件市场规模，归纳成100字"
            " 然后进行邮件发送。"
            " ⼀旦发送完邮件表示你完成了任务。"
            )
        ],
    }
)

#获取最终结果
result = events['messages'][-1].content
result

#查看中间结果
for message in events['messages']:
    print(message.content)
    print("-----------------------------------")

{"searchParameters":{"q":"过去5年AI软件市场规模","hl":"zh-cn","type":"search","engine":"google"},"organic":[{"title":"IDC发布中国人工智能软件及应用市场研究报告","link":"https://yuanzhuo.bnu.edu.cn/article/727","snippet":"中国人工智能产业化应用在过去5年间已经取得显著的成效，呈现出无可比拟的规模与速度：中国人工智能软件市场规模在2020年达到230.9亿元人民币，约为美国AI软件市场规模的6成 ...","position":1},{"title":"[PDF] 中国人工智能产业研究报告（VI）","link":"https://pdf.dfcfw.com/pdf/H3_AP202404191630645407_1.pdf?1713535861000.pdf","snippet":"2023年的中国AI基础数据服务市场规模为37亿元，由上层大模型应用带. 来的数据需求正改变着AI基础数据服务的工作结构。以传统NLP任务为主的分词、词性标注等 ...","date":"2024年4月19日","position":2},{"title":"工业AI 软件市场规模和份额分析- 行业研究报告- 增长趋势","link":"https://www.mordorintelligence.com/zh-CN/industry-reports/industrial-ai-software-market","snippet":"工业AI软件市场分析 今年工业人工智能软件市场价值843.4 亿美元。 预计在预测期内的复合年增长率为35.97%，到未来五年将达到3919.7 亿美元。 更加注重从工业数据中获取价值 ...","position":3},{"title":"贝恩预测：2027年全球AI软硬件市场规模有望达到9900亿美元","link":"https://www.stcn.com/article/detail/1495114.html","snippet":"... 市场预计将以40%至55%的年增长率持续增长，到2027年市场规模将达到7800亿至9900亿美元。生成式AI的潜在市场规模增长速度已经超过了软件